In [1]:
import pandas as pd
from pathlib import Path

import optuna
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.pipeline import Pipeline
from sklearn.metrics import balanced_accuracy_score

C:\miniforge3\envs\machine-learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## preprocessing

In [2]:
DATA_DIR = Path("../data/kaggle-irrigation")
SEED = 42

In [3]:
train = pd.read_csv(DATA_DIR / "train.csv", index_col="id")
train.head()

X = train.drop(columns="Irrigation_Need")
y = train["Irrigation_Need"]

# encode y
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [4]:
# split data
X_train, X_test, y_train, y_test = train_test_split(X, y_encoded, test_size=0.4, random_state=SEED, shuffle=True, stratify=y_encoded)

# reuse the same splits
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

In [5]:
# set up transformation for X
ct = ColumnTransformer(
    [
        ("dummify", OneHotEncoder(), make_column_selector(dtype_include="str"))
    ],
    remainder="passthrough" # keep numerical vars. we don't need to std for RF
)

## base model

In [6]:
base_pipe = Pipeline(
    [
        ("ct", ct),
        ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1))
    ]
)
base_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('ct', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('dummify', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse

In [7]:
y_preds_base = base_pipe.predict(X_test)
balanced_accuracy_score(y_test, y_preds_base)

0.9547207612037153

In [8]:
base_cv_scores = cross_val_score(base_pipe, X_test, y_test, cv=skf, scoring="balanced_accuracy")

print(base_cv_scores)
print(base_cv_scores.mean())

[0.95389525 0.95322254 0.9533362  0.95219998 0.94815482]
0.9521617575014567


## finetuning

In [9]:
def objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 10, 150)
    max_depth = trial.suggest_int("max_depth", 5, 30)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 20)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])

    rf_optuna = RandomForestClassifier(
        random_state=SEED,
        n_jobs=-1,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_features=max_features,
        min_samples_leaf=min_samples_leaf
    )

    optuna_pipe = Pipeline(
        [
            ("ct", ct),
            ("model", rf_optuna)
        ]
    )

    cv_scores = cross_val_score(optuna_pipe, X, y_encoded, cv=skf, scoring="balanced_accuracy")
    return cv_scores.mean()

In [10]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=5)

[I 2026-04-16 17:41:08,698] A new study created in memory with name: no-name-7ac7a47c-e502-4b2d-8610-034a24d099a8
[I 2026-04-16 17:42:14,225] Trial 0 finished with value: 0.9157141769707311 and parameters: {'n_estimators': 113, 'max_depth': 12, 'min_samples_leaf': 18, 'max_features': 'log2'}. Best is trial 0 with value: 0.9157141769707311.
[I 2026-04-16 17:43:46,290] Trial 1 finished with value: 0.9044766188628774 and parameters: {'n_estimators': 129, 'max_depth': 10, 'min_samples_leaf': 17, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.9157141769707311.
[I 2026-04-16 17:45:17,782] Trial 2 finished with value: 0.9511766014483317 and parameters: {'n_estimators': 144, 'max_depth': 29, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 2 with value: 0.9511766014483317.
[I 2026-04-16 17:46:46,949] Trial 3 finished with value: 0.9510315517993512 and parameters: {'n_estimators': 139, 'max_depth': 21, 'min_samples_leaf': 7, 'max_features': 'sqrt'}. Best is trial 2 with val

In [11]:
best_pipe = Pipeline(
    [
        ("ct", ct),
        ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1, **study.best_params))
    ]
)

best_pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('ct', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('dummify', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformers contains sparse

In [12]:
cv_scores = cross_val_score(best_pipe, X, y, cv=skf, scoring="balanced_accuracy")
print(cv_scores)
print(cv_scores.mean())

[0.95053548 0.9519307  0.95264981 0.95207127 0.94869574]
0.9511766014483317


## submission

In [13]:
test = pd.read_csv(DATA_DIR / "test.csv", index_col="id")
test.head()

,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,Wind_Speed_kmh,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region
id,,,,,,,,,,,,,,,,,,,
630000,Silt,6.36,26.19,0.59,2.81,17.83,30.24,1533.38,5.40,3.00,Maize,Sowing,Rabi,Canal,River,13.59,Yes,47.48,West
630001,Clay,5.87,9.88,1.18,3.26,21.18,78.07,576.05,7.22,15.88,Cotton,Sowing,Rabi,Drip,Reservoir,6.12,Yes,56.43,South
630002,Sandy,6.22,26.55,0.96,0.85,26.87,60.35,545.30,9.43,2.63,Wheat,Sowing,Kharif,Sprinkler,Reservoir,3.11,Yes,20.00,East
630003,Clay,7.68,53.58,0.83,0.55,41.74,36.05,1211.03,6.69,1.86,Maize,Harvest,Rabi,Canal,Groundwater,2.27,No,102.99,North
630004,Loamy,5.23,59.02,0.54,2.11,41.08,52.47,1321.91,4.11,5.71,Cotton,Sowing,Kharif,Canal,Groundwater,12.39,Yes,13.33,Central


In [15]:
final_preds = pd.DataFrame({"Irrigation_Need": le.inverse_transform(best_pipe.predict(test))}, index=test.index)
final_preds.index.name = "id"

final_preds.to_csv(DATA_DIR / "bagging_preds.csv")

In [16]:
final_preds

,Irrigation_Need
id,
630000,Low
630001,Low
630002,Low
630003,Low
630004,Low
...,...
899995,Medium
899996,Low
899997,Medium
